In [ ]:
# ================================================================
# CELL 1: Installation + Imports + Config + Data Loading
# Run this FIRST on a fresh runtime
# ================================================================

# --- 1. Install ---
!pip install -q torch torchvision torchaudio
!pip install -q diffusers transformers accelerate peft
!pip install -q clean-fid scikit-learn pandas matplotlib seaborn tqdm

# --- 2. Imports ---
import os, time, random, warnings, json
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torchvision.transforms as T
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, roc_curve
from diffusers import AutoencoderKL, DDPMScheduler, DDIMScheduler, LCMScheduler, UNet2DConditionModel, StableDiffusionPipeline
from transformers import CLIPTextModel, CLIPTokenizer
from peft import LoraConfig, get_peft_model, PeftModel
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# --- 3. Config ---
class Config:
    DATASET_CSV = "./content/vindr_2k.csv"
    IMAGE_ROOT = "./content/images/"
    OUTPUT_DIR = "./content/outputs/"
    SD_MODEL_ID = "runwayml/stable-diffusion-v1-5"
    LCM_LORA_ID = "latent-consistency/lcm-lora-sdv1-5"
    IMAGE_SIZE = 512
    LORA_RANK = 16
    LORA_ALPHA = 32
    FT_EPOCHS = 50
    FT_BATCH_SIZE = 1
    FT_GRAD_ACCUM = 16
    FT_LR = 1e-4
    NORMAL_PROMPT = "a chest xray radiograph of a healthy patient, clear lungs, no abnormalities, grayscale medical image"
    ABNORMAL_PROMPT = "a chest xray radiograph showing pathological findings, abnormal lungs, grayscale medical image"
    NEGATIVE_PROMPT = "blurry, low quality, artifacts, text, watermark, colored, cartoon"
    NUM_SYNTHETIC_PER_CLASS = 500
    GEN_BATCH_SIZE = 4
    CLASSIFIER_SIZE = 224
    CLF_EPOCHS = 30
    CLF_BATCH_SIZE = 32
    CLF_LR = 1e-4
    TRAIN_RATIO = 0.7
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15
    ABLATION_STEPS = [1, 2, 4, 8, 10]
    SEED = 42


cfg = Config()
for d in ['checkpoints', 'synthetic/lcm', 'synthetic/ddim', 'synthetic/ddpm',
          'synthetic/ablation', 'results/plots']:
    os.makedirs(f"{cfg.OUTPUT_DIR}/{d}", exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(cfg.SEED)

# # --- 4. Load CSV + Binary Labels + Splits ---
df = pd.read_csv(cfg.DATASET_CSV)
print(f"\nDataset: {df.shape}")
print(f"Label values: {df['label'].unique()}")
print(f"Label counts:\n{df['label'].value_counts()}\n")

df['binary_label'] = (df['label'] != 'No finding').astype(int)
print(f"Normal (0): {(df['binary_label']==0).sum()}, Abnormal (1): {(df['binary_label']==1).sum()}")
print(f"Ratio: {(df['binary_label']==1).sum()/(df['binary_label']==0).sum():.2f}\n")

train_df, temp_df = train_test_split(df, test_size=(cfg.VAL_RATIO+cfg.TEST_RATIO),
    stratify=df['binary_label'], random_state=cfg.SEED)
val_df, test_df = train_test_split(temp_df, test_size=cfg.TEST_RATIO/(cfg.VAL_RATIO+cfg.TEST_RATIO),
    stratify=temp_df['binary_label'], random_state=cfg.SEED)
print(f"Train: {len(train_df)} {dict(Counter(train_df['binary_label']))}")
print(f"Val:   {len(val_df)} {dict(Counter(val_df['binary_label']))}")
print(f"Test:  {len(test_df)} {dict(Counter(test_df['binary_label']))}\n")

# --- 5. Dataset Classes ---
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['Path']).convert('RGB')
        label = row['binary_label']
        if self.transform:
            img = self.transform(img)
        return img, label

class SyntheticDataset(Dataset):
    def __init__(self, folder, label, transform=None):
        self.paths = sorted([os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(('.png','.jpg'))])
        self.label = label
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.label

train_transform = T.Compose([
    T.Resize((cfg.CLASSIFIER_SIZE, cfg.CLASSIFIER_SIZE)),
    T.RandomHorizontalFlip(0.5), T.RandomRotation(10),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_transform = T.Compose([
    T.Resize((cfg.CLASSIFIER_SIZE, cfg.CLASSIFIER_SIZE)),
    T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_dataset = ChestXrayDataset(train_df, transform=train_transform)
val_dataset = ChestXrayDataset(val_df, transform=val_transform)
test_dataset = ChestXrayDataset(test_df, transform=val_transform)
train_loader = DataLoader(train_dataset, batch_size=cfg.CLF_BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=cfg.CLF_BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=cfg.CLF_BATCH_SIZE, shuffle=False, num_workers=2)

imgs, labels = next(iter(train_loader))
print(f"Batch shape: {imgs.shape}, Labels: {labels[:8].tolist()}")
print("="*50)
print("ALL SETUP COMPLETE")
print("="*50)


Disabling PyTorch because PyTorch >= 2.4 is required but found 2.3.1
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
`CLIPImageProcessor` requires torchvision (not installed); falling back to `CLIPImageProcessorPil` for backward compatibility. Install torchvision to use the default backend, or import `CLIPImageProcessorPil` directly to silence this warning.
`SiglipImageProcessor` requires torchvision (not installed); falling back to `SiglipImageProcessorPil` for backward compatibility. Install torchvision to use the default backend, or import `SiglipImageProcessorPil` directly to silence this warning.


torch: 2.3.1+cpu, CUDA: False

Dataset: (2500, 8)
Label values: ['No finding' 'Aortic enlargement' 'Pleural thickening' 'Cardiomegaly'
 'Pulmonary fibrosis' 'Other abnormality']
Label counts:
label
No finding            1311
Aortic enlargement     604
Cardiomegaly           343
Pulmonary fibrosis     124
Pleural thickening      73
Other abnormality       45
Name: count, dtype: int64

Normal (0): 1311, Abnormal (1): 1189
Ratio: 0.91

Train: 1750 {1: 832, 0: 918}
Val:   375 {0: 196, 1: 179}
Test:  375 {1: 178, 0: 197}



In [ ]:
# ================================================================
# CELL 2: Step 1 — LoRA Fine-tuning of SD 1.5 on VinDr Chest X-rays
# Takes ~2-3 hours on T4 for 50 epochs
# ================================================================

class FinetuneDataset(Dataset):
    def __init__(self, dataframe, tokenizer, size=512):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.transform = T.Compose([
            T.Resize((size, size)), T.ToTensor(), T.Normalize([0.5],[0.5])
        ])
        self.prompts = {0: cfg.NORMAL_PROMPT, 1: cfg.ABNORMAL_PROMPT}
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['Path']).convert('RGB')
        img = self.transform(img)
        prompt = self.prompts[row['binary_label']]
        tokens = self.tokenizer(prompt, padding="max_length", max_length=77,
                                truncation=True, return_tensors="pt")
        return img, tokens.input_ids.squeeze(0)

print("Loading Stable Diffusion 1.5...")
tokenizer = CLIPTokenizer.from_pretrained(cfg.SD_MODEL_ID, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(cfg.SD_MODEL_ID, subfolder="text_encoder").to(device)
vae = AutoencoderKL.from_pretrained(cfg.SD_MODEL_ID, subfolder="vae").to(device)
unet = UNet2DConditionModel.from_pretrained(cfg.SD_MODEL_ID, subfolder="unet")
noise_scheduler = DDPMScheduler.from_pretrained(cfg.SD_MODEL_ID, subfolder="scheduler")

vae.requires_grad_(False); vae.eval()
text_encoder.requires_grad_(False); text_encoder.eval()

lora_config = LoraConfig(
    r=cfg.LORA_RANK, lora_alpha=cfg.LORA_ALPHA,
    target_modules=["to_q", "to_v", "to_k", "to_out.0"],
    lora_dropout=0.05,
)
unet = get_peft_model(unet, lora_config)
unet.to(device)
unet.print_trainable_parameters()

ft_dataset = FinetuneDataset(train_df, tokenizer, size=cfg.IMAGE_SIZE)
ft_loader = DataLoader(ft_dataset, batch_size=cfg.FT_BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
optimizer = optim.AdamW(unet.parameters(), lr=cfg.FT_LR, weight_decay=0.01)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.FT_EPOCHS * len(ft_loader))

print(f"\nFine-tuning: {cfg.FT_EPOCHS} epochs, eff_batch={cfg.FT_BATCH_SIZE*cfg.FT_GRAD_ACCUM}")
print(f"Steps/epoch: {len(ft_loader)}, Total: {cfg.FT_EPOCHS * len(ft_loader)}")

unet.train()
loss_history = []

for epoch in range(cfg.FT_EPOCHS):
    epoch_loss = 0.0
    pbar = tqdm(ft_loader, desc=f"Epoch {epoch+1}/{cfg.FT_EPOCHS}", leave=False)
    for step, (images, input_ids) in enumerate(pbar):
        images, input_ids = images.to(device), input_ids.to(device)
        with torch.no_grad():
            latents = vae.encode(images).latent_dist.sample() * vae.config.scaling_factor
        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                  (latents.shape[0],), device=device).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
        with torch.no_grad():
            encoder_hidden_states = text_encoder(input_ids)[0]
        noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
        loss = F.mse_loss(noise_pred, noise) / cfg.FT_GRAD_ACCUM
        loss.backward()
        if (step + 1) % cfg.FT_GRAD_ACCUM == 0:
            nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
            optimizer.step(); lr_scheduler.step(); optimizer.zero_grad()
        epoch_loss += loss.item() * cfg.FT_GRAD_ACCUM
        pbar.set_postfix(loss=f"{loss.item()*cfg.FT_GRAD_ACCUM:.4f}")
    avg_loss = epoch_loss / len(ft_loader)
    loss_history.append(avg_loss)
    print(f"Epoch {epoch+1}/{cfg.FT_EPOCHS} — Loss: {avg_loss:.4f}")
    if (epoch + 1) % 10 == 0:
        save_path = f"{cfg.OUTPUT_DIR}/checkpoints/lora_epoch_{epoch+1}"
        unet.save_pretrained(save_path)
        print(f"  Checkpoint saved: {save_path}")

final_path = f"{cfg.OUTPUT_DIR}/checkpoints/lora_final"
unet.save_pretrained(final_path)
print(f"\nDone! Final model: {final_path}")

plt.figure(figsize=(8, 4))
plt.plot(loss_history); plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("LoRA Fine-tuning Loss"); plt.grid(True); plt.tight_layout()
plt.savefig(f"{cfg.OUTPUT_DIR}/results/plots/finetune_loss.png", dpi=150)
plt.show()

In [ ]:
# ================================================================
# CELL 3: Step 2 — Generate Synthetic Images (LCM-4, DDIM-50, DDPM-999)
# LCM ~5 min, DDIM ~25 min, DDPM ~2-3 hrs on T4
# ================================================================

print("Loading fine-tuned SD pipeline...")
pipe = StableDiffusionPipeline.from_pretrained(
    cfg.SD_MODEL_ID, torch_dtype=torch.float16, safety_checker=None
)
pipe.unet = PeftModel.from_pretrained(pipe.unet, f"{cfg.OUTPUT_DIR}/checkpoints/lora_final")
pipe.to(device)
print("LoRA weights loaded.\n")

scheduler_config = pipe.scheduler.config

def generate_images(pipe, prompt, negative_prompt, save_dir, num_images, batch_size,
                    scheduler_class, num_steps, guidance_scale, label_name):
    pipe.scheduler = scheduler_class.from_config(scheduler_config)
    os.makedirs(save_dir, exist_ok=True)
    total_time = 0.0
    generated = 0
    print(f"  Generating {num_images} {label_name} images | {num_steps} steps...")
    while generated < num_images:
        current_batch = min(batch_size, num_images - generated)
        torch.cuda.synchronize()
        start = time.time()
        with torch.no_grad():
            result = pipe(
                prompt=[prompt] * current_batch,
                negative_prompt=[negative_prompt] * current_batch,
                num_inference_steps=num_steps,
                guidance_scale=guidance_scale,
                height=cfg.IMAGE_SIZE, width=cfg.IMAGE_SIZE,
            )
        torch.cuda.synchronize()
        elapsed = time.time() - start
        total_time += elapsed
        for i, img in enumerate(result.images):
            img.save(f"{save_dir}/{label_name}_{generated + i:04d}.png")
        generated += current_batch
        print(f"    {generated}/{num_images} ({elapsed:.1f}s)")
    return total_time

# Run LCM first (fastest), then DDIM, then DDPM
methods = {
    "lcm": {"scheduler": LCMScheduler, "steps": 4, "guidance": 1.0},
    "ddim": {"scheduler": DDIMScheduler, "steps": 50, "guidance": 7.5},
    "ddpm": {"scheduler": DDPMScheduler, "steps": 999, "guidance": 7.5},
}

timing_results = {}

for method_name, mcfg in methods.items():
    print(f"\n{'='*50}")
    print(f"METHOD: {method_name.upper()} ({mcfg['steps']} steps)")
    print(f"{'='*50}")
    save_base = f"{cfg.OUTPUT_DIR}/synthetic/{method_name}"
    t_normal = generate_images(
        pipe, cfg.NORMAL_PROMPT, cfg.NEGATIVE_PROMPT,
        f"{save_base}/normal", cfg.NUM_SYNTHETIC_PER_CLASS,
        cfg.GEN_BATCH_SIZE, mcfg['scheduler'], mcfg['steps'], mcfg['guidance'], "normal"
    )
    t_abnormal = generate_images(
        pipe, cfg.ABNORMAL_PROMPT, cfg.NEGATIVE_PROMPT,
        f"{save_base}/abnormal", cfg.NUM_SYNTHETIC_PER_CLASS,
        cfg.GEN_BATCH_SIZE, mcfg['scheduler'], mcfg['steps'], mcfg['guidance'], "abnormal"
    )
    total = t_normal + t_abnormal
    total_imgs = cfg.NUM_SYNTHETIC_PER_CLASS * 2
    timing_results[method_name] = {
        "total_time_sec": total, "time_per_image": total / total_imgs,
        "steps": mcfg['steps'], "total_images": total_imgs
    }
    print(f"\n  {method_name.upper()}: {total:.1f}s total, {total/total_imgs:.2f}s/image")

# Speed comparison
print(f"\n{'='*60}")
print(f"GENERATION SPEED COMPARISON")
print(f"{'='*60}")
print(f"{'Method':<10} {'Steps':<8} {'Total':<14} {'Per Image':<12} {'Speedup'}")
print(f"{'-'*60}")
ddpm_tpi = timing_results['ddpm']['time_per_image']
for name, res in timing_results.items():
    speedup = ddpm_tpi / res['time_per_image']
    print(f"{name.upper():<10} {res['steps']:<8} {res['total_time_sec']:<14.1f} {res['time_per_image']:<12.2f} {speedup:.1f}x")

with open(f"{cfg.OUTPUT_DIR}/results/timing_results.json", 'w') as f:
    json.dump(timing_results, f, indent=2)

# Show sample images
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for col, mn in enumerate(['lcm', 'ddim', 'ddpm']):
    for row, lb in enumerate(['normal', 'abnormal']):
        img = Image.open(f"{cfg.OUTPUT_DIR}/synthetic/{mn}/{lb}/{lb}_0000.png")
        axes[row, col].imshow(img); axes[row, col].set_title(f"{mn.upper()} - {lb.title()}")
        axes[row, col].axis('off')
plt.suptitle("Sample Generated Images", fontsize=14); plt.tight_layout()
plt.savefig(f"{cfg.OUTPUT_DIR}/results/plots/sample_generations.png", dpi=150)
plt.show()

In [ ]:
# ================================================================
# ADS (Adaptive Diffusion Sampling) generation
# ================================================================
ADS_STEPS = 25
GUIDANCE_RANGE = (3.0, 9.0)   # Random guidance range
ETA = 0.4                     # Controls randomness

def generate_ads(pipe, prompt, neg_prompt, save_dir, num_images, label_name, batch_size=4):

    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    os.makedirs(save_dir, exist_ok=True)

    total_time = 0.0
    generated = 0

    while generated < num_images:
        bs = min(batch_size, num_images - generated)

        guidance = random.uniform(*GUIDANCE_RANGE)

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.time()

        with torch.no_grad():
            result = pipe(
                prompt=[prompt] * bs,
                negative_prompt=[neg_prompt] * bs,
                num_inference_steps=ADS_STEPS,
                guidance_scale=guidance,
                eta=ETA,
                height=cfg.IMAGE_SIZE,
                width=cfg.IMAGE_SIZE,
            )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed = time.time() - start
        total_time += elapsed

        for i, img in enumerate(result.images):
            img.save(f"{save_dir}/{label_name}_{generated + i:04d}.png")

        generated += bs

        if generated % 100 == 0:
            print(f"{generated}/{num_images} | guidance={guidance:.2f}")

    return total_time

In [ ]:
# ================================================================
# Generate synthetic NORMAL and ABNORMAL images
# ================================================================
print("Generating ADS images...")

generate_ads(
    pipe,
    cfg.NORMAL_PROMPT,
    cfg.NEGATIVE_PROMPT,
    f"{cfg.OUTPUT_DIR}/synthetic/ads/normal",
    cfg.NUM_SYNTHETIC_PER_CLASS,
    "normal"
)

generate_ads(
    pipe,
    cfg.ABNORMAL_PROMPT,
    cfg.NEGATIVE_PROMPT,
    f"{cfg.OUTPUT_DIR}/synthetic/ads/abnormal",
    cfg.NUM_SYNTHETIC_PER_CLASS,
    "abnormal"
)

print("Generation complete.")

In [1]:
# ================================================================
# BUILD ADS LOADER (CLEAN)
# ================================================================

# Synthetic data paths (ADS-generated)
ads_syn_normal = SyntheticDataset(
    f"{cfg.OUTPUT_DIR}/synthetic/ads/normal",
    label=0,
    transform=train_transform
)

ads_syn_abnormal = SyntheticDataset(
    f"{cfg.OUTPUT_DIR}/synthetic/ads/abnormal",
    label=1,
    transform=train_transform
)

# ✅ Safety check
assert len(ads_syn_normal) > 0, "No ADS normal images found!"
assert len(ads_syn_abnormal) > 0, "No ADS abnormal images found!"

# Combine synthetic datasets
ads_synthetic = ConcatDataset([ads_syn_normal, ads_syn_abnormal])

# Create ADS dataset (real + synthetic)
ads_dataset = ConcatDataset([train_dataset, ads_synthetic])

# Final ADS loader
ads_loader = DataLoader(
    ads_dataset,
    batch_size=cfg.CLF_BATCH_SIZE,   # ✅ FIXED
    shuffle=True,
    num_workers=2
)

print(f"ADS Loader Ready: {len(ads_dataset)} samples")

NameError: name 'SyntheticDataset' is not defined

In [ ]:
# ================================================================
# CELL 4: Train DenseNet-121 on 3 Data Variants (LCM REMOVED)
# ================================================================

def build_classifier():
    model = models.densenet121(weights='IMAGENET1K_V1')
    model.classifier = nn.Linear(model.classifier.in_features, 2)
    return model.to(device)

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    correct, total = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    acc = correct / total
    auc = roc_auc_score(all_labels, all_probs)
    f1 = f1_score(all_labels, (np.array(all_probs) > 0.5).astype(int))
    return acc, auc, f1, np.array(all_probs), np.array(all_labels)


def train_full(model, train_loader, val_loader, variant_name):
    print(f"\n🚀 Starting training: {variant_name}")

    labels_list = [y for _, y in train_loader.dataset]
    counts = Counter(labels_list)

    weights = torch.tensor([
        len(labels_list)/(2*counts[0]),
        len(labels_list)/(2*counts[1])
    ]).to(device)

    criterion = nn.CrossEntropyLoss(
        weight=weights,
        label_smoothing=0.1
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.CLF_LR,  # ✅ FIXED
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.5,
        patience=3
    )

    best_auc = 0
    best_state = None

    history = {
        'train_loss': [],
        'val_auc': []
    }

    from tqdm import tqdm
    pbar = tqdm(range(cfg.CLF_EPOCHS), desc=f"{variant_name}", ncols=100)  # ✅ FIXED

    for epoch in pbar:

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_acc, val_auc, val_f1, _, _ = evaluate(model, val_loader)

        scheduler.step(val_auc)

        history['train_loss'].append(train_loss)
        history['val_auc'].append(val_auc)

        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        pbar.set_postfix({
            "loss": f"{train_loss:.4f}",
            "val_auc": f"{val_auc:.4f}",
            "lr": f"{optimizer.param_groups[0]['lr']:.1e}"
        })

    model.load_state_dict(best_state)
    model.to(device)

    print(f"✅ Best Val AUC ({variant_name}): {best_auc:.4f}")

    return model, history


# ======================
# Build data variants
# ======================
print("Building data variants...\n")

print("Variant A: Real only")
loader_a = train_loader

def make_augmented_loader(method_name):
    syn_normal = SyntheticDataset(f"{cfg.OUTPUT_DIR}/synthetic/{method_name}/normal", label=0, transform=train_transform)
    syn_abnormal = SyntheticDataset(f"{cfg.OUTPUT_DIR}/synthetic/{method_name}/abnormal", label=1, transform=train_transform)

    # ✅ SAFETY CHECK
    assert len(syn_normal) > 0, f"No images in {method_name}/normal"
    assert len(syn_abnormal) > 0, f"No images in {method_name}/abnormal"

    combined = ConcatDataset([train_dataset, syn_normal, syn_abnormal])

    print(f"Real: {len(train_dataset)} + Synthetic: {len(syn_normal)} + {len(syn_abnormal)} = {len(combined)}")

    return DataLoader(combined, batch_size=cfg.CLF_BATCH_SIZE, shuffle=True, num_workers=2)  # ✅ FIXED


print("Variant B: Real + DDPM")
loader_b = make_augmented_loader("ddpm")

print("Variant C: Real + DDIM")
loader_c = make_augmented_loader("ddim")

print("Variant D: ADS")

# ✅ SAFETY CHECK
assert 'ads_loader' in globals(), "ADS loader not defined!"
loader_d = ads_loader 


# ======================
# Train all variants
# ======================
results = {}
variants = [
    ("real_only", loader_a),
    ("real+ddpm", loader_b),
    ("real+ddim", loader_c),
    ("ads", loader_d),
]

# ✅ ensure results folder exists
os.makedirs(f"{cfg.OUTPUT_DIR}/results", exist_ok=True)

for variant_name, loader in variants:
    print(f"\n{'='*50}")
    print(f"Training: {variant_name.upper()}")
    print(f"{'='*50}")

    seed_everything(cfg.SEED)

    model = build_classifier()
    model, history = train_full(model, loader, val_loader, variant_name)

    test_acc, test_auc, test_f1, test_probs, test_labels = evaluate(model, test_loader)

    results[variant_name] = {
        'accuracy': test_acc,
        'auc': test_auc,
        'f1': test_f1
    }

    print(f"TEST — AUC: {test_auc:.4f}, Acc: {test_acc:.4f}, F1: {test_f1:.4f}")

    # ✅ safer filename
    variant_safe = variant_name.replace("+", "_")
    torch.save(model.state_dict(), f"{cfg.OUTPUT_DIR}/checkpoints/clf_{variant_safe}.pth")


# ======================
# Final Results
# ======================
print(f"\n{'='*60}")
print("FINAL RESULTS — CLASSIFIER COMPARISON")
print(f"{'='*60}")

for name, res in results.items():
    print(f"{name:<15} AUC: {res['auc']:.4f}, Acc: {res['accuracy']:.4f}, F1: {res['f1']:.4f}")

with open(f"{cfg.OUTPUT_DIR}/results/classifier_results.json", 'w') as f:
    json.dump(results, f, indent=2)

In [ ]:
# ================================================================
# CELL 8: Final Summary — CLEAN (NO LCM, WITH ADS)
# ================================================================

import json

with open(f"{cfg.OUTPUT_DIR}/results/classifier_results.json") as f:
    clf_results = json.load(f)

with open(f"{cfg.OUTPUT_DIR}/results/fid_results.json") as f:
    fid_data = json.load(f)

# Optional timing (only DDPM/DDIM if available)
try:
    with open(f"{cfg.OUTPUT_DIR}/results/timing_results.json") as f:
        timing = json.load(f)
except:
    timing = {
        "ddim": {"time_per_image": 0.0, "steps": 50},
        "ddpm": {"time_per_image": 0.0, "steps": 999},
    }

print("WARNING: No timing file found. Update timing numbers manually.\n")


# ================================================================
# TABLE 1 (NO LCM, ADD ADS)
# ================================================================
print("=" * 80)
print("TABLE 1: Classifier Performance")
print("=" * 80)

print(f"{'Variant':<15} {'AUC':<10} {'Acc':<10} {'F1':<10} {'FID':<10}")

for variant, method_key in [
    ('real_only', None),
    ('real+ddpm', 'ddpm'),
    ('real+ddim', 'ddim'),
    ('ads', None)   # ADS added
]:
    r = clf_results[variant]

    # FID only for diffusion methods
    fid_val = fid_data['main'].get(method_key, '-') if method_key else '-'
    fid_str = f"{fid_val:.1f}" if isinstance(fid_val, (int, float)) else '-'

    print(f"{variant:<15} {r['auc']:<10.4f} {r['accuracy']:<10.4f} {r['f1']:<10.4f} {fid_str:<10}")


# ================================================================
# KEY FINDINGS (UPDATED)
# ================================================================
real_auc = clf_results['real_only']['auc']
ddpm_auc = clf_results['real+ddpm']['auc']
ddim_auc = clf_results['real+ddim']['auc']
ads_auc = clf_results['ads']['auc']

print(f"\n{'='*60}")
print("KEY FINDINGS")
print("=" * 60)

print(f"1. Diffusion improvement (DDPM): {real_auc:.4f} -> {ddpm_auc:.4f} (+{ddpm_auc-real_auc:.4f})")
print(f"2. Diffusion improvement (DDIM): {real_auc:.4f} -> {ddim_auc:.4f} (+{ddim_auc-real_auc:.4f})")
print(f"3. ADS improvement: {real_auc:.4f} -> {ads_auc:.4f} (+{ads_auc-real_auc:.4f})")
print(f"4. Best method: {'ADS' if ads_auc > max(ddpm_auc, ddim_auc) else 'Diffusion'}")


# ================================================================
# COMBINED PLOT (NO LCM, ADD ADS)
# ================================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ---- AUC Comparison ----
ax = axes[0]

bar_names = ['Real\nOnly', 'DDPM', 'DDIM', 'ADS']
bar_aucs = [
    clf_results['real_only']['auc'],
    clf_results['real+ddpm']['auc'],
    clf_results['real+ddim']['auc'],
    clf_results['ads']['auc']
]

bars = ax.bar(bar_names, bar_aucs, edgecolor='black')

for bar, val in zip(bars, bar_aucs):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.002, f'{val:.3f}', ha='center')

ax.set_ylim(min(bar_aucs)-0.04, max(bar_aucs)+0.02)
ax.set_ylabel('Test AUC')
ax.set_title('Classifier Performance Comparison')
ax.grid(axis='y', alpha=0.3)


# ---- FID Comparison ----
ax = axes[1]

fid_names = ['DDPM', 'DDIM']
fid_vals = [
    fid_data['main'].get('ddpm', 0),
    fid_data['main'].get('ddim', 0)
]

bars = ax.bar(fid_names, fid_vals, edgecolor='black')

for bar, val in zip(bars, fid_vals):
    ax.text(bar.get_x()+bar.get_width()/2, val+1, f'{val:.1f}', ha='center')

ax.set_ylabel('FID (lower=better)')
ax.set_title('Synthetic Image Quality')
ax.grid(axis='y', alpha=0.3)


plt.tight_layout()
plt.savefig(f"{cfg.OUTPUT_DIR}/results/final_summary.png", dpi=200)
plt.show()

print(f"\nAll results saved to {cfg.OUTPUT_DIR}/results/")
print("PROJECT COMPLETE!")